In [1]:
from ultralytics import YOLO
import cv2

from sort.sort import Sort

import numpy as np
import pandas as pd

In [19]:
log_df = pd.DataFrame(columns=["id", "x1", "y1", "x2", "y2", "class", "confidence"])

In [27]:
model = YOLO("yolov8n.pt")

In [21]:
tracker = Sort(max_age=30, min_hits=3, iou_threshold=0.3)

In [28]:
cap = cv2.VideoCapture(1)

In [29]:
ret, frame = cap.read()

print(frame.shape)
xbuffer = 10
ybuffer = 8
roi = [xbuffer, ybuffer, frame.shape[1]-xbuffer, frame.shape[0]-ybuffer]


(720, 1280, 3)


In [24]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLOv8 inference
    results = model.predict(source=frame, show=False, conf=0.3)


    # Get first result (single frame)
    result = results[0]

    detections = np.empty((0, 5))

    # Extract bounding boxes
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0]  
        x1, y1, x2, y2 = int(x1), int(x2), int(y1), int(y2)

        conf = box.conf[0]           
        cls = int(box.cls[0])

        CurrentArray = np.array([x1, y1, x2, y2, conf])
        detections = np.vstack((detections, CurrentArray))

    TrackResults = tracker.update(detections)
    for tracking in TrackResults:
        x1, y1, x2, y2, id = tracking
        x1, y1, x2, y2 = int(x1), int(x2), int(y1), int(y2)

        cx, cy = (x2-x1)/2, (y2-y1)/2

        if roi[0]< cx < roi[2] and roi[1] < cy < roi[3] and int(id) not in log_df['id'].values:
            log_df.loc[len(log_df)] = [id, x1, y1, x2, y2, model.names[cls], float(conf)]

        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 3)
        cv2.putText(frame, f'{id}', (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 1)
        # print(track)
    
    # Display result
    cv2.imshow("YOLOv8 Webcam with Boxes", frame)

    # Exit on ' '
    if cv2.waitKey(1) & 0xFF == ord(' '):
        break

cap.release()
cv2.destroyAllWindows()


0: 576x1024 (no detections), 286.1ms
Speed: 12.7ms preprocess, 286.1ms inference, 1.1ms postprocess per image at shape (1, 3, 576, 1024)

0: 576x1024 (no detections), 255.4ms
Speed: 11.7ms preprocess, 255.4ms inference, 1.2ms postprocess per image at shape (1, 3, 576, 1024)

0: 576x1024 (no detections), 246.6ms
Speed: 11.6ms preprocess, 246.6ms inference, 1.1ms postprocess per image at shape (1, 3, 576, 1024)

0: 576x1024 (no detections), 224.4ms
Speed: 13.9ms preprocess, 224.4ms inference, 1.1ms postprocess per image at shape (1, 3, 576, 1024)

0: 576x1024 3 targets, 223.3ms
Speed: 10.4ms preprocess, 223.3ms inference, 1.9ms postprocess per image at shape (1, 3, 576, 1024)

0: 576x1024 6 targets, 218.3ms
Speed: 12.2ms preprocess, 218.3ms inference, 2.3ms postprocess per image at shape (1, 3, 576, 1024)

0: 576x1024 5 targets, 210.1ms
Speed: 13.0ms preprocess, 210.1ms inference, 1.7ms postprocess per image at shape (1, 3, 576, 1024)

0: 576x1024 4 targets, 208.3ms
Speed: 12.2ms prepro

In [18]:
print(model.names)

{0: 'target'}


In [150]:
print(log_df)

       id  x1   y1   x2   y2         class  confidence
0  4768.0  23   77  537  479        person    0.852019
1  4769.0  27   71  537  479        person    0.859774
2  4770.0  23   68  543  479        person    0.869325
3  4919.0   1   25  270  475        person    0.691459
4  5039.0   3  271   87  487        person    0.899150
5  5093.0  30  276  314  474        person    0.568963
6  5204.0   0  132   80  183  dining table    0.520930
